# NB03 — Statistical Model & Visualization

**Environment:** Local Python (no Spark)

**Purpose:** From the cached biome-coverage matrix, run the statistical tests and produce the paper figures.

**Inputs:**
- `data/biome_coverage_matrix.csv` (from NB02)
- `data/species_biome_assignment.csv` (from NB02)

**Outputs:**
- `figures/NB03_biome_heatmap.png` — biome × KEGG-module coverage-rate heatmap
- `figures/NB03_biome_pdb_rate_bars.png` — biome-level PDB-direct rate with bootstrap CIs
- `figures/NB03_af_vs_pdb_scatter.png` — AF-confident-only rate vs PDB-covered rate per biome
- `data/biome_gap_rank.csv` — biomes ranked by PDB-direct deficit
- `data/logistic_model_results.csv` — coefficients + cluster-robust SEs

**Tests:**
- Fisher's exact per biome × function cell for PDB-direct enrichment/depletion vs. global rate (BH-FDR corrected)
- Logistic regression: `pdb_hit ~ biome + phylum + kegg_module + is_core + log(cluster_size)` with cluster-robust SEs by species

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests

## Load

In [ ]:
matrix = pd.read_csv("data/biome_coverage_matrix.csv")
species_env = pd.read_csv("data/species_biome_assignment.csv")
print(matrix.shape, species_env.shape)

## Fisher enrichment per (biome × KEGG) cell

In [ ]:
# Global PDB-direct rate
global_direct = matrix["n_pdb_direct"].sum()
global_total  = matrix["n_clusters"].sum()
global_rate   = global_direct / global_total

def fisher_row(r):
    a = r["n_pdb_direct"]
    b = r["n_clusters"] - a
    c = global_direct - a
    d = global_total  - global_direct - b
    if b < 0 or c < 0 or d < 0:
        return np.nan
    _, p = stats.fisher_exact([[a, b], [c, d]])
    return p

matrix["fisher_p"] = matrix.apply(fisher_row, axis=1)
matrix["fisher_q_bh"] = multipletests(matrix["fisher_p"].fillna(1.0), method="fdr_bh")[1]
matrix["pdb_direct_rate"] = matrix["n_pdb_direct"] / matrix["n_clusters"]
matrix.to_csv("data/biome_coverage_matrix_annotated.csv", index=False)

## Biome-level rankings

In [ ]:
biome_summary = matrix.groupby("compartment").agg(
    n_clusters=("n_clusters", "sum"),
    n_pdb_direct=("n_pdb_direct", "sum"),
    n_pdb_homolog=("n_pdb_homolog", "sum"),
    n_af_confident=("n_af_confident", "sum"),
).reset_index()
biome_summary["pdb_direct_rate"] = biome_summary["n_pdb_direct"] / biome_summary["n_clusters"]
biome_summary["pdb_any_rate"]    = (biome_summary["n_pdb_direct"] + biome_summary["n_pdb_homolog"]) / biome_summary["n_clusters"]
biome_summary["af_conf_rate"]    = biome_summary["n_af_confident"] / biome_summary["n_clusters"]

def bootstrap_ci(k, n, iters=1000, alpha=0.05):
    if n == 0: return (np.nan, np.nan)
    p = k / n
    draws = np.random.binomial(n, p, size=iters) / n
    return np.quantile(draws, [alpha/2, 1-alpha/2])

biome_summary[["ci_lo", "ci_hi"]] = biome_summary.apply(
    lambda r: pd.Series(bootstrap_ci(r["n_pdb_direct"], r["n_clusters"])), axis=1
)
biome_summary.sort_values("pdb_direct_rate").to_csv("data/biome_gap_rank.csv", index=False)
biome_summary.sort_values("pdb_direct_rate")

## Figure 1 — Biome PDB-direct rate with CIs

In [ ]:
bs = biome_summary.sort_values("pdb_direct_rate")
fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(bs))))
ax.errorbar(
    bs["pdb_direct_rate"], bs["compartment"],
    xerr=[bs["pdb_direct_rate"] - bs["ci_lo"], bs["ci_hi"] - bs["pdb_direct_rate"]],
    fmt="o", capsize=3,
)
ax.axvline(global_rate, color="grey", linestyle="--", label=f"global rate = {global_rate:.3f}")
ax.set_xlabel("PDB-direct rate (per gene cluster)")
ax.set_title("Structural coverage gap by biome")
ax.legend()
fig.tight_layout()
fig.savefig("figures/NB03_biome_pdb_rate_bars.png", dpi=150)

## Figure 2 — Heatmap: biome × top-N KEGG modules

In [ ]:
top_kos = (matrix.groupby("kegg_orthology_id")["n_clusters"].sum()
           .sort_values(ascending=False).head(50).index)
sub = matrix[matrix["kegg_orthology_id"].isin(top_kos)].copy()
pivot = sub.pivot_table(
    index="kegg_orthology_id", columns="compartment",
    values="pdb_direct_rate", aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(8, 12))
sns.heatmap(pivot, cmap="viridis", ax=ax, cbar_kws={"label": "PDB-direct rate"})
ax.set_title("Coverage rate by biome × top-50 KEGG orthologs")
fig.tight_layout()
fig.savefig("figures/NB03_biome_heatmap.png", dpi=150)

## Figure 3 — AF-confident-only rate vs PDB-any rate per biome (H2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(biome_summary["pdb_any_rate"], biome_summary["af_conf_rate"])
for _, r in biome_summary.iterrows():
    ax.annotate(r["compartment"], (r["pdb_any_rate"], r["af_conf_rate"]), fontsize=8)
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("PDB coverage rate (direct + homolog)")
ax.set_ylabel("AF-confident rate (MSA depth ≥ 300)")
ax.set_title("Where does AF overstate coverage? (H2)")
fig.tight_layout()
fig.savefig("figures/NB03_af_vs_pdb_scatter.png", dpi=150)

## Logistic model (H1, H3)

Requires per-cluster-level table with biome + phylum + is_core + cluster_size. Load `cluster_biome_coverage.parquet` locally (needs re-materialization from NB02 output; consider a smaller-sample precheck first). Skipping full run here — see follow-up NB03b for the model.

For now, aggregate-level model: PDB-direct-rate ~ biome + KEGG module + is_core, weighted by n_clusters, using GLM binomial with `n_pdb_direct / n_clusters` as the outcome.

In [ ]:
model_df = matrix.dropna(subset=["kegg_orthology_id", "compartment"]).copy()
model_df["y"] = model_df["n_pdb_direct"] / model_df["n_clusters"]

X = pd.get_dummies(model_df[["compartment", "is_core"]], drop_first=True).astype(float)
X = sm.add_constant(X)

glm = sm.GLM(model_df["y"], X, family=sm.families.Binomial(),
             freq_weights=model_df["n_clusters"]).fit()
print(glm.summary())
pd.DataFrame({"coef": glm.params, "se": glm.bse, "pvalue": glm.pvalues}) \
    .to_csv("data/logistic_model_results.csv")